# IMPORT STATEMENTS
Below are the different imports for the demo, both from classes created for the project and for external libraries.

In [ ]:
from classes.neural_network import NeuralNetwork, FeedForwardLayer
from classes.trainer import Trainer
from classes.recommender import Recommender

import numpy as np
import matplotlib.pyplot as plt

# NEURAL NETWORK CREATION DEMO
Below is a demo for how to create a neural network using the NeuralNetwork class of this project. We are creating a deep autoencoder here with 4 encoder layers and 4 decoder layers. The embedding generated by the encoder section in this network is 32 dimensional.

In [ ]:
bias_scale = 0.0001
nn = NeuralNetwork(
    input_size=202,
    output_size=202
)
# Layer 1
layer_1_weights = Trainer.get_he_initialization(202, 165)
layer_1_biases = np.random.rand(165) * bias_scale
layer_1 = FeedForwardLayer(layer_1_weights, layer_1_biases, NeuralNetwork.relu, NeuralNetwork.relu_dx)
# Layer 2
layer_2_weights = Trainer.get_he_initialization(165, 128)
layer_2_biases = np.random.rand(128) * bias_scale
layer_2 = FeedForwardLayer(layer_2_weights, layer_2_biases, NeuralNetwork.relu, NeuralNetwork.relu_dx)
# Layer 3
layer_3_weights = Trainer.get_he_initialization(128, 64)
layer_3_biases = np.random.rand(64) * bias_scale
layer_3 = FeedForwardLayer(layer_3_weights, layer_3_biases, NeuralNetwork.relu, NeuralNetwork.relu_dx)
# Layer 4
layer_4_weights = Trainer.get_he_initialization(64, 32)
layer_4_biases = np.random.rand(32) * bias_scale
layer_4 = FeedForwardLayer(layer_4_weights, layer_4_biases, NeuralNetwork.relu, NeuralNetwork.relu_dx)
# Layer 5
layer_5_weights = Trainer.get_he_initialization(32, 64)
layer_5_biases = np.random.rand(64) * bias_scale
layer_5 = FeedForwardLayer(layer_5_weights, layer_5_biases, NeuralNetwork.relu, NeuralNetwork.relu_dx)
# Layer 6
layer_6_weights = Trainer.get_he_initialization(64, 128)
layer_6_biases = np.random.rand(128) * bias_scale
layer_6 = FeedForwardLayer(layer_6_weights, layer_6_biases, NeuralNetwork.relu, NeuralNetwork.relu_dx)
# Layer 7
layer_7_weights = Trainer.get_he_initialization(128, 165)
layer_7_biases = np.random.rand(165) * bias_scale
layer_7 = FeedForwardLayer(layer_7_weights, layer_7_biases, NeuralNetwork.relu, NeuralNetwork.relu_dx)
# Layer 8
layer_8_weights = Trainer.get_he_initialization(165, 202)
layer_8_biases = np.random.rand(202) * bias_scale
layer_8 = FeedForwardLayer(layer_8_weights, layer_8_biases, NeuralNetwork.relu, NeuralNetwork.relu_dx)
# Add each layer
nn.append_layer(layer_1)
nn.append_layer(layer_2)
nn.append_layer(layer_3)
nn.append_layer(layer_4)
nn.append_layer(layer_5)
nn.append_layer(layer_6)
nn.append_layer(layer_7)
nn.append_layer(layer_8)

# TRAINING DEMO
Below is a demo of the model training for 2 epochs.

In [ ]:
# Create trainer object
trainer = Trainer(
    training_network=nn,
    initial_lr=0.001,
    final_lr=0.0001,
    num_epochs=2,
    dataset_path='./MillionSongSpotifyTracksDataset',
    output_folder='./networks/demo_networks'
)

trainer.train_model()

# SOME DATA FROM THE DATASET
Here is a small taste of the dataset used for training.

In [ ]:
# get first 5 file paths
print("Displaying first 5 songs in dataset...")
file_paths = trainer.get_file_paths()[:5]
for file_path in file_paths:
    song = trainer.get_song_data_from_file(file_path)
    print()
    print(song.to_string())

# LATENT SPACE TRAVERSAL DEMO
Here is a small demo of the latent space traversal of the Recommender class. Two keep visualization simple, a set of 2D points that are the corners of a hexagon are used.

In [ ]:
curr_track = ('a', np.array([1, 1]))
tracks = [
    ('f', np.array([-1, 1])),
    ('c', np.array([1, -1])),
    ('b', np.array([2, 0])),
    ('e', np.array([-2, 0])),
    ('d', np.array([-1, -1])),
]

# test initializing recommendations
recommender = Recommender()
recommender.set_seed_track_info(curr_track)
recommender.init_selected_track_ids(tracks.copy())
recommender.init_recommendations()
recommendation_path_x = [i[1][0] for i in recommender._recommendations]
recommendation_path_y = [i[1][1] for i in recommender._recommendations]
plt.plot(recommendation_path_x, recommendation_path_y)
plt.scatter([1], [1])
plt.show()
print("Recommendations path after initialization:")
print(recommender._recommendations)
print()

# test getting next recommendation when previous recommendation taken
next_rec_id = recommender.get_next_recommendation_track_id(('b', np.array([2, 0])))
recommendation_path_x = [i[1][0] for i in recommender._recommendations]
recommendation_path_y = [i[1][1] for i in recommender._recommendations]
plt.plot(recommendation_path_x, recommendation_path_y)
plt.scatter([2], [0])
plt.show()
print("Next recommendation given user played b:")
print(next_rec_id)
print("Recommendation path:")
print(recommender._recommendations)
print()

# test getting next recommendation when previous recommendation NOT taken
next_rec_id = recommender.get_next_recommendation_track_id(tracks[0])
recommendation_path_x = [i[1][0] for i in recommender._recommendations]
recommendation_path_y = [i[1][1] for i in recommender._recommendations]
plt.plot(recommendation_path_x, recommendation_path_y)
plt.scatter([-1], [1])
plt.show()
print("Next recommendation given user played f:")
print(next_rec_id)
print("Recommendation path:")
print(recommender._recommendations)

# LOADING A NEURAL NETWORK FROM A FILE
Here is how a neural network is loaded.

In [ ]:
encoder = NeuralNetwork.load_network('networks/encoder/encoder.pkl')

# GENERATION OF EMBEDDING EXAMPLE
Here is an example of how an embedding is generated.

In [ ]:
# get song data for a song
file_path = trainer.get_file_paths()[0]
song = trainer.get_song_data_from_file(file_path)
nn_input = song.get_nn_input()

# feed forward through encoder
encoder.set_input(nn_input)
encoder.feed_forward()
output = encoder.get_output()

# print output
print("Embedding:")
print(output)